# Annotated review copy

**Environment checked against:** Python 3.7 (all cells parsed with the 3.7 grammar).
**Syntax errors found: 0.** No walrus operators, `match`, f-string `=` or other post-3.7 syntax is used, so every cell compiles on 3.7.
Note: the notebook's kernel metadata says `3.14.7` - update it if you really run 3.7.

**Marker legend (inline comments, original code is untouched):**
- `BUG` - wrong result or crash
- `RISK` - works on the test data but fails on edge cases / design weakness
- `STYLE` - readability / naming / convention

**Where the real problems are (ranked):**
1. Cell 32 `ArrayBasedLinkedList` - aliased nodes + 5 further bugs
2. Cell 29 `CQueue` - wrong head/tail convention, `count` never updated
3. Cells 34-35 BSTs - no node class, `postorder` broken, recursive `inorder`/`preorder` crash
4. Cell 31 `LinkedList` - `insert_sorted` and `delete` bugs
5. Cell 27 `Stack` - `pop(val)` signature, `peek` on empty
6. Cells 3, 5, 6 conversions; cells 10/12/13 `False` vs index `0`
7. Sorting cells 15-24 are correct (typo `haore` only)

**Cross-cutting readability points:** shadowing built-ins (`bin`, `hex`, `chr`); mixing `False` as a "failure" value with legitimate results (`0`); no docstrings; inconsistent spacing after commas; a few misleading names (`mid` for a partition index).

The corrected version is in the companion notebook `review_corrected.ipynb`.

# Data Representation

## Binary Numbers

**Cell 2 - `dec_to_bin`** | 0 syntax errors | 1 risk, 2 style
Logic is correct for non-negative integers. Readability: add a docstring; avoid naming a variable `bin`.

In [ ]:
# Denary/Decimal to Binary
def dec_to_bin(dec):   # STYLE: no docstring; parameter/local names shadow built-ins (see below)
    bin = ""   # STYLE: `bin` shadows the built-in bin(); rename (e.g. `bits`)

    while dec:   # RISK: a negative input never terminates (-1 // 2 == -1 forever); validate dec >= 0
        bin = str(dec%2)+bin
        dec //= 2

    return bin or "0"

**Cell 3 - `bin_to_dec`** | 1 bug
Crashes on any non-empty input because it adds a string to an int. Readability: `bin`/`chr` shadow built-ins.

In [ ]:
# Binary to Denary/Decimal
def bin_to_dec(bin):   # STYLE: `bin` shadows built-in
    dec = 0

    for chr in bin:   # STYLE: `chr` shadows built-in chr()
        dec *= 2
        dec += chr   # BUG: chr is the str '0'/'1' -> TypeError (int += str). Use int(chr)

    return dec

## Hexadecimals

**Cell 5 - `dec_to_hex`** | 1 bug
Wrong result for 0. Readability: `hex` shadows a built-in.

In [ ]:
# Denary/Decimal to Hexadecimal
def dec_to_hex(dec):
    hex = ""   # STYLE: `hex` shadows built-in hex()
    hex_digits = "0123456789ABCDEF"

    while dec:
        hex = hex_digits[dec%16] + hex
        dec //= 16

    return hex   # BUG: dec_to_hex(0) returns '' (dec_to_bin(0) returns '0'). Use `or "0"`

**Cell 6 - `hex_to_dex`** | 1 bug, 1 risk
Typo in function name and no validation of digits.

In [ ]:
# Hexadecimal to Denary/Decimal
def hex_to_dex(hex):   # BUG: name typo (dex -> dec); `hex` also shadows a built-in
    dec = 0
    hex_digits = "0123456789ABCDEF"

    for chr in hex:   # STYLE: `chr` shadows built-in
        dec *= 16
        dec += hex_digits.find(chr)   # RISK: find() gives -1 for lowercase/invalid digits and silently corrupts the result; use .index() after .upper()

    return dec

# Fundamental Algorithms

**Cell 8 - test data** | style only
Fine. Deriving the expected list with `sorted()` removes the risk of the two lists disagreeing.

In [ ]:
unsorted_test_lst = [12,1231,1,23,36,453,2234,3125,43654,4,786,243,13,4,65,3376,35687,33,2]   # STYLE: hard-coded expected result can drift from the input; derive it with sorted(unsorted_test_lst)
test_lst = [1, 2, 4, 4, 12, 13, 23, 33, 36, 65, 243, 453, 786, 1231, 2234, 3125, 3376, 35687, 43654]

## Linear Search

**Cell 10 - `linear_search`** | 1 bug
The `False`/`0` problem is real in this very cell: `linear_search(test_lst, 1)` prints `0`, and `linear_search(test_lst, 6767)` prints `False` - and `0 == False`, so `if linear_search(...)` treats a hit at index 0 as a miss.

In [ ]:
# Linear Search
def linear_search(array,seek):   # STYLE: PEP 8 - space after commas (array, seek)
    
    for i in range(len(array)):
        if array[i] == seek:
            return i
        
    return False   # BUG: False == 0 in Python, so 'not found' looks identical to 'found at index 0' (compare the 2nd print). Return -1 or None

print(linear_search(test_lst,6767))
print(linear_search(test_lst,1))
print(linear_search(test_lst,43654))

## Binary Search

**Cell 12 - iterative binary search** | 1 bug
Algorithm is correct (requires a sorted list). Same return-value problem as cell 10.

In [ ]:
# Binary Search - Iterative
def binary_search_iter(array,seek):
    lo = 0
    hi = len(array)-1

    while lo <= hi:
        mid = (lo + hi) // 2
        if array[mid] == seek:
            return mid
        elif seek > array[mid]:
            lo = mid+1
        else:
            hi = mid-1

    return False   # BUG: same False-vs-index-0 ambiguity as linear_search. Return -1

print(binary_search_iter(test_lst,6767))
print(binary_search_iter(test_lst,1))
print(binary_search_iter(test_lst,43654))

**Cell 13 - recursive binary search** | 1 bug
Algorithm is correct. Same return-value problem. The `lo=None, hi=None` default pattern is a good way to hide the extra parameters.

In [ ]:
# Binary Search - Recursive
def binary_search_rec(array,seek,lo=None,hi=None):
    if lo is None and hi is None:
        lo = 0
        hi = len(array)-1

    if lo > hi:
        return False   # BUG: same False-vs-index-0 ambiguity. Return -1 (recursion itself is correct)
    else:
        mid = (lo + hi) // 2

        if seek == array[mid]:
            return mid
        elif seek > array[mid]:
            return binary_search_rec(array,seek,mid+1,hi)
        else:
            return binary_search_rec(array,seek,lo,mid-1)

print(binary_search_rec(test_lst,6767))
print(binary_search_rec(test_lst,1))
print(binary_search_rec(test_lst,43654))

## Bubble Sort

**Cell 15 - bubble sort** | no issues
Correct, with a sensible early-exit flag. Verified against the test list.

In [ ]:
# Bubble Sort
def bubble_sort(array):
    for i in range(len(array)-1):
        swap = False
        for j in range(len(array)-i-1):
            if array[j+1] < array[j]:
                array[j+1],array[j] = array[j],array[j+1]
                swap = True
        if not swap:
            break

bubble_lst = unsorted_test_lst[:]
bubble_sort(bubble_lst)
print(bubble_lst)
print(bubble_lst == test_lst)

## Insertion Sort

**Cell 17 - insertion sort** | style only
Correct (swap-based variant). Verified.

In [ ]:
# Insertion Sort
def insertion_sort(array):
    for i in range(len(array)):   # STYLE: i = 0 does nothing; start at range(1, len(array))
        j = i
        while j > 0 and array[j-1] > array[j]:
            array[j-1],array[j] = array[j],array[j-1]
            j-=1

insertion_lst = unsorted_test_lst[:]
insertion_sort(insertion_lst)
print(insertion_lst)
print(insertion_lst == test_lst)

## Merge Sort

**Cell 19 - merge sort** | 1 risk
Correct and clean; only stability is affected. Verified.

In [ ]:
# Merge Sort
def merge_sort(array):
    if len(array) <= 1:
        return array

    mid = len(array) // 2

    left = merge_sort(array[:mid])
    right = merge_sort(array[mid:])

    return merge(left,right)

def merge(left,right):
    merged = []
    i = j = 0

    while i < len(left) and j < len(right):
        if left[i] < right[j]:   # RISK: not stable - on ties the RIGHT item is taken first. Use <=
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    merged += left[i:] + right[j:]

    return merged

merge_lst = merge_sort(unsorted_test_lst)
print(merge_lst)
print(merge_lst == test_lst)

## Quick Sort

**Cell 21 - quick sort (list-comprehension)** | style
Correct; the three-way split handles duplicates well. Verified.

In [ ]:
# Quick Sort
def quick_sort(array):
    if len(array) <= 0:   # STYLE: base case should be <= 1 (works, but makes needless extra calls)
        return array

    import random   # STYLE: import at the top of the cell/module, not inside a recursive function
    pivot = random.choice(array)

    lesser = [i for i in array if i < pivot]   # RISK: not in-place - builds 3 new lists per call (O(n) extra space per level)
    equal = [i for i in array if i == pivot ]
    greater = [i for i in array if i > pivot]

    return quick_sort(lesser) + equal + quick_sort(greater)

quick_lst = quick_sort(unsorted_test_lst)
print(quick_lst)
print(quick_lst == test_lst)

## Quick Sort - In Place

**Cell 23 - Lomuto quick sort** | 1 risk, 1 style
Correct on the test data. Verified.

In [ ]:
# Quick Sort In Place - Lomuto's Partition
def quick_sort_lomuto(array):
    return _quick_sort_lomuto(array,0,len(array)-1)
        
def _quick_sort_lomuto(array,lo,hi):
    if lo < hi:
        mid = lomuto_partition(array,lo,hi)   # STYLE: misleading name - this is the pivot's final index, not a midpoint
        _quick_sort_lomuto(array,lo,mid-1)
        _quick_sort_lomuto(array,mid+1,hi)
    
def lomuto_partition(array,lo,hi):
    pivot = array[hi]   # RISK: O(n^2) and recursion depth n on already-sorted input (RecursionError past ~1000 items). Use a random / median-of-3 pivot
    i = lo - 1
    for j in range(lo,hi):
        if array[j] < pivot:
            i += 1
            array[i],array[j] = array[j],array[i]
    array[hi],array[i+1] = array[i+1],array[hi]

    return i+1

lomuto_lst = unsorted_test_lst[:]
quick_sort_lomuto(lomuto_lst)
print(lomuto_lst)
print(lomuto_lst == test_lst)

**Cell 24 - Hoare quick sort** | 1 bug (typo), 1 risk
The partition logic is correct (returns `j`, recurses on `[lo, mid]` and `[mid+1, hi]`). The `i += 1` / inner `while` pairing imitates a do-while; a comment would help readers.

In [ ]:
# Quick Sort In Place - Hoare's Partition
def quick_sort_haore(array):   # BUG: name typo (haore -> hoare)
    return _quick_sort_hoare(array,0,len(array)-1)

def _quick_sort_hoare(array,lo,hi):
    if lo < hi:
        mid = hoare_partition(array,lo,hi)   # STYLE: misleading name - it's the partition split index
        _quick_sort_hoare(array,lo,mid)
        _quick_sort_hoare(array,mid+1,hi)

def hoare_partition(array,lo,hi):
    i = lo - 1
    j = hi + 1

    pivot = array[lo]   # RISK: same sorted-input worst case as Lomuto

    while True:
        i += 1
        while array[i] < pivot:
            i += 1

        j -= 1
        while array[j] > pivot:
            j -= 1

        if i < j:
            array[i],array[j] = array[j],array[i]

        else:
            return j

hoare_lst = unsorted_test_lst[:]
quick_sort_haore(hoare_lst)   # BUG: calls the misspelt name (fix both together)
print(hoare_lst)
print(hoare_lst == test_lst)

# Abstract Data Types

## Stack

**Cell 27 - Stack** | 2 bugs, 1 risk
`push` and `is_full` / `is_empty` are fine.

In [ ]:
# Stack
class Stack:
    def __init__(self,size):
        self.size = size
        self.array = [None]*size
        self.tail = 0

    def is_empty(self):
        return self.tail == 0

    def is_full(self):
        return self.tail == self.size

    def push(self,val):
        if self.is_full():
            return False
        self.array[self.tail] = val
        self.tail += 1   # RISK: push() returns None on success but False on failure - inconsistent; return True

    def pop(self):   # BUG: stray `val` parameter -> stack.pop() raises TypeError
        if self.is_empty():
            return False
        self.tail -= 1
        return self.array[self.tail]

    def peek(self):
        return self.array[self.tail-1]   # BUG: on an empty stack tail-1 == -1, so this returns the last array slot instead of signalling empty

## Queue

**Cell 29 - Circular queue** | 3 bugs - the most broken class
The head/tail convention is inconsistent and `count` is never updated. Simplest fix: `head` = index of the front item, `tail` = next free slot, both start at 0.

In [ ]:
# Circular Queue
class CQueue:
    def __init__(self,size):
        self.size = size
        self.count = 0
        self.q = [None] * size
        self.head = 0
        self.tail = -1   # BUG: first enqueue writes q[-1] (the LAST slot) - after enqueue(1), enqueue(2) the array is [2, None, 1]

    def is_empty(self):
        return self.count == 0
    
    def is_full(self):
        return self.count == self.size
    
    def enqueue(self,val):
        if self.is_full():
            return False

        self.q[self.tail] = val
        self.tail = (self.tail + 1) % self.size   # BUG: self.count is never incremented -> is_empty() always True, is_full() never True

    def dequeue(self):
        if self.is_empty():
            return False

        self.head = (self.head + 1) % self.size   # BUG: head moves BEFORE the read (skips the front item) and count is never decremented
        return self.q[self.head]

    def display(self):
        if self.is_empty():
            return False

        return [self.q[(self.head+i+1)%self.size] for i in range(0,self.count)]   # RISK: only valid for a 'head is one before front' convention that enqueue/dequeue don't follow

## Linked List

**Cell 31 - Linked list** | 3 bugs
`insert_first` and `insert_last` are fine.

In [ ]:
# Linked List (Sorted) + Recursive Operations   # STYLE: comment promises 'Recursive Operations' but none exist; insert_first/insert_last also break the sorted invariant
class LinkedListNode:
    def __init__(self,data):
        self.data = data
        self.next = None

class LinkedList:
    def __init__(self):
        self.root = None

    def insert_first(self,node):
        node.next = self.root
        self.root = node

    def insert_last(self,node):
        if self.root is None:
            self.root = node

        else:
            curr = self.root
            while curr.next is not None:   # BUG: stops before the last node, so the tail can never be deleted. Loop on `curr is not None`
                curr = curr.next

            curr.next = node

    def insert_sorted(self,node):
        if self.root is None or node < self.root.data:   # BUG: compares a node object with data -> TypeError. Use node.data. Also use <= (see below)
            node.next = self.root
            self.root = node

        else:
            prev = None
            curr = self.root
            while curr is not None and curr.data < node.data:
                prev = curr
                curr = curr.next

            prev.next = node   # BUG: if node.data == root.data the loop doesn't advance, prev is still None -> AttributeError
            node.next = curr

    def delete(self,data):
        if self.root is None:
            return False   # STYLE: returns False when empty but None otherwise - pick one convention (True/False)
        
        elif self.root.data == data:
            self.root = self.root.next
            
        else:
            prev = None
            curr = self.root
            while curr.next is not None:
                if curr.data == data:
                    prev.next = curr.next
                    return
                prev = curr
                curr = curr.next
    
            

**Cell 32 - Array-based linked list** | 6 bugs - needs the most rework
Free-list idea is right (`free` chain through unused slots, `root = -1` for empty).

In [ ]:
# Array Based Sorted Linked List

class ArrayBasedLinkedListNode:
    def __init__(self,data):
        self.data = data
        self.next = -1

class ArrayBasedLinkedList:
    def __init__(self,size):
        self.array = [ArrayBasedLinkedListNode(None)]*size   # BUG: `*size` repeats ONE node object - every slot aliases the same node. Use a list comprehension
        for i in range(size-1):
            self.array[i].next = i+1
        self.root = -1
        self.free = 0

    def is_full(self):
        return self.free == -1

    def insert_sorted(self,data):
        idx = self.free   # BUG: no is_full() guard - when free == -1 this uses array[-1] and corrupts the list
        node = self.array[idx]
        node.data = data
        self.free = node.next

        if self.root == -1 or self.array[self.root].data > data:   # BUG: must be >=; with equal data the else-branch leaves prev == -1 and writes to array[-1]
            node.next = self.root
            self.root = idx

        else:
            prev = -1
            curr = self.root
            
            while curr != -1 and self.array[curr].data < data:
                prev = curr
                curr = self.array[curr].next

            self.array[prev].next = idx
            node.next = curr

    def delete(self,data):
        if self.root == -1:
            return False
        if self.array[self.root].data == data:
            self.root = self.array[self.root].next   # BUG: root is overwritten before the old index is saved, so the NEW root is put on the free list
            self.array[self.root].next = self.free
            self.free = self.root
        else:
            prev = -1
            curr = self.root

            while curr != -1:
                if self.array[curr].data != data:   # BUG: condition inverted - should be ==
                    self.array[prev] = self.array[curr].next   # BUG: replaces a whole node with an int; should assign .next
                    self.array[curr].next = self.free
                    self.free = curr
                    return
                prev = curr
                curr = self.array[curr].next

## Binary Search Tree

**Cell 34 - BST (iterative)** | 3 bugs
`insert`, `search`, `inorder`, `preorder` are correct once a node class exists. `postorder` cannot run as written.

In [ ]:
# Binary Search Tree - Iterative
class BinarySearchTreeIter:   # BUG: insert() needs node objects (.data/.left/.right) but no node class exists in the notebook -> unusable as given. Add TreeNode
    def __init__(self):
        self.root = None

    def insert(self,node):
        if self.root is None:
            self.root = node

        else:
            curr = self.root
            while True:
                if node.data > curr.data:
                    if curr.right is None:
                        curr.right = node
                        return
                    curr = curr.right
                else:
                    if curr.left is None:
                        curr.left = node
                        return
                    curr = curr.left

    def search(self,data):
        if self.root is None:
            return False

        curr = self.root
        while curr is not None:
            if curr.data == data:
                return curr
            elif data > curr.data:
                curr = curr.right
            else:
                curr = curr.left

        return False   # STYLE: inorder returns False for empty but a list otherwise - return [] (same for preorder/postorder)

    def inorder(self):
        if self.root is None:
            return False
        result = []    
        stack = []
        curr = self.root


        while curr is not None or stack:

            while curr is not None:
                stack.append(curr)
                curr = curr.left

            curr = stack.pop()
            result.append(curr.data)

            curr = curr.right

        return result

    def preorder(self):
        if self.root is None:
            return False
        result = []
        stack = [self.root]

        while stack:
            curr = stack.pop()
            result.append(curr.data)

            if curr.left is not None:
                stack.append(curr.left)   # BUG: should push `curr`, not curr.left (pushes None / wrong node)

            if curr.right is not None:
                stack.append(curr.right)

        return result


    def postorder(self):
        if self.root is None:
            return False
        
        result = []        
        stack = []
        curr = self.root
        last = None

        while curr is not None or stack:

            while curr is not None:
                stack.append(curr.left)
                curr = curr.left

            peek = stack[-1]

            if peek.right is not None and last is not peek.right:
                curr = peek.right

            else:
                result.append(peek.data)
                last = stack.pop   # BUG: missing () - stores the bound method and never pops -> infinite loop

        return result

**Cell 35 - BST (recursive)** | 1 bug (4 occurrences)
`insert`, `search`, `postorder` are correct. Note recursion depth equals tree height, so a skewed tree hits the recursion limit.

In [ ]:
# Binary Search Tree - Recursive
class BinarySearchTreeRec:
    def __init__(self):
        self.root = None

    def insert(self,node):
        self.root = self._insert(node,self.root)

    def _insert(self,node,root):
        if root is None:
            return node

        if node.data > root.data:
            root.right = self._insert(node,root.right)
        else:
            root.left = self._insert(node,root.left)

        return root

    def search(self,data):
        return self._search(data,self.root)

    def _search(self,data,root):
        if root is None:
            return False

        if root.data == data:
            return root
        elif data > root.data:
            return self._search(data,root.right)
        else:
            return self._search(data,root.left)

    def inorder(self):
        result = []
        self._inorder(self.root,result)
        return result

    def _inorder(self,root,result):
        if root:
            self._inorder(root.left)   # BUG: missing `result` argument -> TypeError
            result.append(root.data)
            self._inorder(root.right)   # BUG: missing `result` argument

    def preorder(self):
        result = []
        self._preorder(self.root,result)
        return result

    def _preorder(self,root,result):
        if root:
            result.append(root.data)
            self._preorder(root.left)   # BUG: missing `result` argument
            self._preorder(root.right)   # BUG: missing `result` argument

    def postorder(self):
        result = []
        self._postorder(self.root,result)
        return result

    def _postorder(self,root,result):
        if root:
            self._postorder(root.left,result)
            self._postorder(root.right,result)
            result.append(root.data)

## Hash Table

**Cell 37 - Hash table, linear probing** | 1 bug, 2 risks
Probe loop and wrap-around logic are correct, including the full-table check.

In [ ]:
# Hash Table (Linear Probing)
class HashTableLP:
    def __init__(self,size):
        self.size = size
        self.buckets = [None]*size

    def _hash(self,key):
        return hash(key)%self.size   # RISK: str hashes are randomised per process (PYTHONHASHSEED) - fine in memory, never persist bucket positions

    def insert(self,key,value):
        idx = ref = self._hash(key)

        while self.buckets[idx] is not None:   # RISK: an existing key is inserted again instead of updated; no delete (would need tombstones)
            idx += 1
            idx %= self.size

            if idx == ref:
                return False

        self.buckets[idx] = (key,value)

        return True

    def search(self,key):
        idx = ref = self._hash(key)

        while self.buckets[idx] is not None:
            if self.buckets[idx][0] == key:
                return self.buckets[idx][1]
            idx += 1
            idx %= self.size

            if idx == ref:
                return False   # BUG: False == 0, so a stored value of 0/False is indistinguishable from 'not found'. Return None

        return False

**Cell 38 - Hash table, separate chaining** | 1 bug, 1 risk, 1 style
Structure is correct.

In [ ]:
# Hash Table (Separate Chaining)
class HashTableSC:
    def __init__(self,size):
        self.size = size
        self.buckets = [[] for _ in range(size)]

    def _hash(self,key):
        return hash(key) % self.size

    def insert(self,key,value):
        idx = self._hash(key)

        chain = self.buckets[idx]
        chain.append((key,value))   # RISK: existing key is appended again instead of updated

    def search(self,key):
        idx = self._hash(key)

        chain = self.buckets[idx]

        curr = 0   # STYLE: manual index loop; `for k, v in chain:` is clearer
        while curr < len(chain):
            if chain[curr][0] == key:
                return chain[curr][1]
            curr += 1

        return False   # BUG: False == 0 ambiguity (see HashTableLP). Return None